In [21]:
import os
import pandas as pd
import numpy as np

raw_dir = "data/raw"
processed_dir = "data/processed"

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

data = {
    "age": [34, 45, 29, 50, 38, np.nan, 41],
    "income": [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    "score": [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    "zipcode": ["90210", "10001", "60614", "94103", "73301", "12345", "94105"],
    "city": ["Beverly", "New York", "Chicago", "SF", "Austin", "Unknown", "San Francisco"],
    "extra_data": [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan]
}

df = pd.DataFrame(data)

csv_path = os.path.join(raw_dir, "sample_data.csv")

if not os.path.exists(csv_path):
    df.to_csv(csv_path, index=False)
    print(f"sample dataset --> {csv_path}")
else:
    print(f"file exist at {csv_path}. skipping CSV creation to avoid overwrite.")

file exist at data/raw/sample_data.csv. skipping CSV creation to avoid overwrite.


In [22]:
import pathlib
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [23]:
RAW_DIR = pathlib.Path("data/raw")
PROC_DIR = pathlib.Path("data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR.resolve())
print("PROC_DIR:", PROC_DIR.resolve())

RAW_DIR: /Users/that_bat/bootcamp_moshi_wang/homework/homework06/data/raw
PROC_DIR: /Users/that_bat/bootcamp_moshi_wang/homework/homework06/data/processed


In [24]:
csv_path = os.path.join(raw_dir, "sample_data.csv")
df = pd.read_csv(csv_path)

print("shape:", df.shape)
df.head()

shape: (7, 6)


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN


In [25]:
print("data types:")
print(df.dtypes)

print("\nmissing")
print(df.isna().sum())

data types:
age           float64
income        float64
score         float64
zipcode         int64
city              str
extra_data    float64
dtype: object

missing
age           1
income        3
score         1
zipcode       0
city          0
extra_data    5
dtype: int64


In [26]:
def fill_missing_median(df, columns=None):
    df_copy = df.copy()

    if columns is None:
        columns = df_copy.select_dtypes(include=np.number).columns

    for col in columns:
        df_copy[col] = df_copy[col].fillna(df_copy[col].median())

    return df_copy

In [27]:
filled_df = fill_missing_median(
    df,
    columns=["age", "income", "score"]
)

print(filled_df)
print("\nMissing after fill:")
print(filled_df.isna().sum())

    age   income  score  zipcode           city  extra_data
0  34.0  55000.0  0.820    90210        Beverly         NaN
1  45.0  52000.0  0.910    10001       New York        42.0
2  29.0  42000.0  0.805    60614        Chicago         NaN
3  50.0  58000.0  0.760    94103             SF         NaN
4  38.0  52000.0  0.880    73301         Austin         NaN
5  39.5  52000.0  0.650    12345        Unknown         5.0
6  41.0  49000.0  0.790    94105  San Francisco         NaN

Missing after fill:
age           0
income        0
score         0
zipcode       0
city          0
extra_data    5
dtype: int64


In [28]:
def drop_missing(df, columns=None, threshold=None):
    df_copy = df.copy()

    if columns is not None:
        return df_copy.dropna(subset=columns)

    if threshold is not None:
        return df_copy.dropna(
            thresh=int(threshold * df_copy.shape[1])
        )

    return df_copy.dropna()

In [29]:
dropped_df = drop_missing(df, columns=["extra_data"])

print(dropped_df)
print("\nShape before:", df.shape)
print("Shape after:", dropped_df.shape)

    age  income  score  zipcode      city  extra_data
1  45.0     NaN   0.91    10001  New York        42.0
5   NaN     NaN   0.65    12345   Unknown         5.0

Shape before: (7, 6)
Shape after: (2, 6)


In [30]:
def normalize_data(df, columns=None, method="minmax"):
    df_copy = df.copy()

    if columns is None:
        columns = df_copy.select_dtypes(include=np.number).columns

    if method == "minmax":
        scaler = MinMaxScaler()
    else:
        scaler = StandardScaler()

    df_copy[columns] = scaler.fit_transform(df_copy[columns])

    return df_copy

In [31]:
normalized_df = normalize_data(
    filled_df,
    columns=["age", "income", "score"],
    method="minmax"
)

print(normalized_df)
print("\nMin values:")
print(normalized_df[["age", "income", "score"]].min())

print("\nMax values:")
print(normalized_df[["age", "income", "score"]].max())

        age  income     score  zipcode           city  extra_data
0  0.238095  0.8125  0.653846    90210        Beverly         NaN
1  0.761905  0.6250  1.000000    10001       New York        42.0
2  0.000000  0.0000  0.596154    60614        Chicago         NaN
3  1.000000  1.0000  0.423077    94103             SF         NaN
4  0.428571  0.6250  0.884615    73301         Austin         NaN
5  0.500000  0.6250  0.000000    12345        Unknown         5.0
6  0.571429  0.4375  0.538462    94105  San Francisco         NaN

Min values:
age       0.0
income    0.0
score     0.0
dtype: float64

Max values:
age       1.0
income    1.0
score     1.0
dtype: float64


In [32]:
clean_df = df.copy()

clean_df = fill_missing_median(
    clean_df,
    columns=["age", "income", "score"]
)

clean_df = drop_missing(
    clean_df,
    columns=["city", "zipcode"]
)

clean_df = normalize_data(
    clean_df,
    columns=["age", "income", "score"],
    method="minmax"
)

print(clean_df)

        age  income     score  zipcode           city  extra_data
0  0.238095  0.8125  0.653846    90210        Beverly         NaN
1  0.761905  0.6250  1.000000    10001       New York        42.0
2  0.000000  0.0000  0.596154    60614        Chicago         NaN
3  1.000000  1.0000  0.423077    94103             SF         NaN
4  0.428571  0.6250  0.884615    73301         Austin         NaN
5  0.500000  0.6250  0.000000    12345        Unknown         5.0
6  0.571429  0.4375  0.538462    94105  San Francisco         NaN


In [33]:
print("original shape:", df.shape)
print("cleaned", clean_df.shape)

print("\noriginal data type")
print(df.dtypes)

print("\ncleaned")
print(clean_df.dtypes)

print("\nmissing value after cleaning:")
print(clean_df.isna().sum())

original shape: (7, 6)
cleaned (7, 6)

original data type
age           float64
income        float64
score         float64
zipcode         int64
city              str
extra_data    float64
dtype: object

cleaned
age           float64
income        float64
score         float64
zipcode         int64
city              str
extra_data    float64
dtype: object

missing value after cleaning:
age           0
income        0
score         0
zipcode       0
city          0
extra_data    5
dtype: int64


In [34]:
output_path = PROC_DIR / "cleaned_data.csv"

clean_df.to_csv(output_path, index=False)

print("cleaned data saved in", output_path)

cleaned data saved in data/processed/cleaned_data.csv


In [35]:
from src.cleaning import fill_missing_median, drop_missing, normalize_data
print(fill_missing_median.__module__)
print(drop_missing.__module__)
print(normalize_data.__module__)

src.cleaning
src.cleaning
src.cleaning


In [36]:
final_df = df.copy()

final_df = fill_missing_median(
    final_df,
    columns=["age", "income", "score"]
)

final_df = drop_missing(
    final_df,
    columns=["city", "zipcode"]
)

final_df = normalize_data(
    final_df,
    columns=["age", "income", "score"],
    method="minmax"
)

print(final_df)

        age  income     score  zipcode           city  extra_data
0  0.238095  0.8125  0.653846    90210        Beverly         NaN
1  0.761905  0.6250  1.000000    10001       New York        42.0
2  0.000000  0.0000  0.596154    60614        Chicago         NaN
3  1.000000  1.0000  0.423077    94103             SF         NaN
4  0.428571  0.6250  0.884615    73301         Austin         NaN
5  0.500000  0.6250  0.000000    12345        Unknown         5.0
6  0.571429  0.4375  0.538462    94105  San Francisco         NaN


In [37]:
print("final shape:", final_df.shape)

print("\ndata type")
print(final_df.dtypes)

print("\nmissing")
print(final_df.isna().sum())

print("\nvalue range")
print("age:", final_df["age"].min(), final_df["age"].max())
print("income:", final_df["income"].min(), final_df["income"].max())
print("score:", final_df["score"].min(), final_df["score"].max())

final shape: (7, 6)

data type
age           float64
income        float64
score         float64
zipcode         int64
city              str
extra_data    float64
dtype: object

missing
age           0
income        0
score         0
zipcode       0
city          0
extra_data    5
dtype: int64

value range
age: 0.0 1.0
income: 0.0 1.0
score: 0.0 1.0


In [38]:
output_path = PROC_DIR / "cleaned_data.csv"
final_df.to_csv(output_path, index=False)
print("cleaned data saved in", output_path)

cleaned data saved in data/processed/cleaned_data.csv
